# SentinelRAG — 01: Document Ingestion & Chunking

Welcome to the first experimentation notebook for **SentinelRAG (Self-Evaluating Enterprise RAG)**.

### Objectives in this Notebook:
1. **Load Enterprise Knowledge**: Ingest multi-format documents (Markdown, PDF, DOCX, TXT) from `data/raw/`.
2. **Extract Granular Metadata**: Track source file names, sections, pages, and document identifiers.
3. **Experiment with Chunking**: Compare chunk split strategies and token sizing.
4. **Serialize Processed Knowledge**: Save the enriched chunk records to `data/processed/chunks.json` for vector and BM25 indexing.

In [1]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
ROOT_DIR = Path("..").resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from rich import print as rprint
from rich.table import Table
from src.ingestion.loaders import load_documents_from_directory, SentinelDocument
from src.ingestion.chunking import chunk_documents, save_chunks_to_json, ChunkingStrategy
from config.settings import settings

print(f"Project root: {ROOT_DIR}")
print(f"Raw data directory: {settings.raw_data_dir}")

Project root: C:\Users\Keshav\RAG\SentinelRAG
Raw data directory: C:\Users\Keshav\RAG\SentinelRAG\data\raw


## 1. Load Documents from `data/raw/`

We load all available enterprise policies. Each document is parsed and assigned structured metadata.

In [2]:
raw_docs = load_documents_from_directory(settings.raw_data_dir)
print(f"Total parsed document sections/pages: {len(raw_docs)}\n")

# Display summary table of loaded documents
table = Table(title="SentinelRAG — Raw Document Ingestion Summary")
table.add_column("Source File", style="cyan", no_wrap=True)
table.add_column("Section / Page", style="magenta")
table.add_column("Type", style="green")
table.add_column("Characters", justify="right")

for doc in raw_docs:
    section_info = doc.metadata.get("section") or f"Page {doc.metadata.get('page', 1)}"
    table.add_row(
        doc.metadata.get("source", "Unknown"),
        str(section_info),
        doc.metadata.get("file_type", ""),
        str(len(doc.page_content))
    )

rprint(table)

Total parsed document sections/pages: 13



                                 SentinelRAG — Raw Document Ingestion Summary                                 
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━┓
┃ Source File           ┃ Section / Page                                                 ┃ Type ┃ Characters ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━┩
│ IT_Security_Policy.md │ Sentinel Global — Information Security & Acceptable Use Policy │ .md  │        219 │
│ IT_Security_Policy.md │ 1. Scope & Objective                                           │ .md  │        169 │
│ IT_Security_Policy.md │ 2. Authentication & Credential Standards                       │ .md  │        684 │
│ IT_Security_Policy.md │ 3. Device & Endpoint Security                                  │ .md  │        885 │
│ Leave_Policy.md       │ Sentinel Global — Employee Leave & Time-Off Policy             │ .md  │        190 │
│ Leave_Policy.md       │ 1. Overview                                                    │ .md  │        201 │
│ Leave_Policy.md       │ 2. Categories of Leave                                         │ .md  │       1399 │
│ Travel_Policy.md      │ Sentinel Global — Corporate Travel & Expense Policy            │ .md  │        203 │
│ Travel_Policy.md      │ 1. Purpose & Scope                                             │ .md  │        307 │
│ Travel_Policy.md      │ 2. Accommodation & Lodging Guidelines                          │ .md  │        928 │
│ Travel_Policy.md      │ 3. Daily Per Diem & Meals                                      │ .md  │        290 │
│ Travel_Policy.md      │ 4. Transportation & Commute                                    │ .md  │        557 │
│ Travel_Policy.md      │ 5. Expense Submission & Reimbursement Deadlines                │ .md  │        336 │
└───────────────────────┴────────────────────────────────────────────────────────────────┴──────┴────────────┘

## 2. Inspect Ground Truth Policies & Negative Constraints

Let's inspect the key section in `Travel_Policy.md` concerning accommodation caps (₹3,000) and city tiers.

In [3]:
travel_sections = [d for d in raw_docs if "Travel_Policy" in d.source and "Accommodation" in str(d.metadata.get("section", ""))]
if travel_sections:
    rprint("[bold yellow]Inspected Section Sample:[/bold yellow]")
    print(travel_sections[0].page_content)
else:
    print("Travel policy sample:", raw_docs[0].page_content[:400])

Inspected Section Sample:

## 2. Accommodation & Lodging Guidelines

### 2.1 Hotel Accommodation Limits
Employees travelling for approved business purposes can claim hotel accommodation up to **₹3,000 per night** (taxes inclusive). 

* **Uniform Cap:** This limit of ₹3,000 per night applies uniformly across all domestic travel locations, cities, and regions. No exceptions or higher city tiers (including metropolitan zones such as Mumbai, Delhi NCR, or Bengaluru) are authorized under standard travel limits. Any booking exceeding ₹3,000 per night requires prior written sign-off from the Chief Financial Officer (CFO).
* **Booking Window:** Employees must book lodging at least 7 days prior to departure through the corporate travel desk.

### 2.2 Shared Accommodation
When two employees of the same gender travel together on company assignment, they are encouraged to share twin-occupancy rooms with an enhanced combined cap of ₹4,500 per night.

---


## 3. Chunking Experiments

We test chunking parameters to evaluate granularity vs. semantic cohesion:
- Chunk Size: 500 characters
- Chunk Overlap: 80 characters

In [4]:
chunks = chunk_documents(
    documents=raw_docs,
    chunk_size=500,
    chunk_overlap=80,
    strategy=ChunkingStrategy.RECURSIVE
)

print(f"Generated {len(chunks)} enriched chunk records.\n")

# Display sample chunk with full metadata
sample_chunk = chunks[0]
rprint(f"[bold green]Sample Chunk ID:[/bold green] {sample_chunk.chunk_id}")
rprint(f"[bold cyan]Metadata:[/bold cyan] {sample_chunk.metadata}")
rprint(f"[bold yellow]Content:[/bold yellow]\n{sample_chunk.text}")

Generated 26 enriched chunk records.



Sample Chunk ID: IT_Security_Policy_c001

Metadata: {'source': 'IT_Security_Policy.md', 'file_path': 
'C:\\Users\\Keshav\\RAG\\SentinelRAG\\data\\raw\\IT_Security_Policy.md', 'file_type': '.md', 'section': 'Sentinel 
Global — Information Security & Acceptable Use Policy', 'section_index': 1, 'total_sections': 4, 'chunk_id': 
'IT_Security_Policy_c001', 'chunk_index': 1, 'sub_split_index': 1, 'char_length': 219, 'estimated_tokens': 54}

Content:
# Sentinel Global — Information Security & Acceptable Use Policy

**Document ID:** POL-SEC-2024-001  
**Effective Date:** February 15, 2024  
**Version:** 4.0  
**Owner:** Chief Information Security Office (CISO)  

---

## 4. Persist Processed Chunks

Now we write the enriched chunks into `data/processed/chunks.json` so Phase 2 (Vector Search + BM25) can load them instantly.

In [5]:
output_file = settings.processed_data_dir / "chunks.json"
save_chunks_to_json(chunks, output_file)
print(f"Successfully saved {len(chunks)} chunks to {output_file.resolve()}")

Successfully saved 26 chunks to C:\Users\Keshav\RAG\SentinelRAG\data\processed\chunks.json
